# Average recorded output tokens across six model arms

This notebook uses only evaluation artifacts that directly record `generation.output_tokens`. The aligned Qwen teacher and aligned-Qwen-trained Llama are excluded because their released result files do not contain generation token counts. Token counts remain tokenizer-specific, so comparisons are grouped into Qwen and Llama families.

In [ ]:
from collections import defaultdict
import hashlib
import json
from pathlib import Path
from statistics import fmean, median

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "PLAN.md").is_file() and (candidate / "experiment").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the mats12 repository root.")


def read_jsonl(path: Path):
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON at {path}:{line_number}") from exc


ROOT = find_repo_root(Path.cwd().resolve())
MAX_NEW_TOKENS = 1_024

MODEL_SPECS = [
    {
        "label": "Abliterated Qwen 3.5 9B\nteacher", "family": "Qwen",
        "path": "runs/behavioral-probe-qwen-20260827T0110Z/raw/qwen.jsonl",
        "expected_model": "huihui-ai/Huihui-Qwen3.5-9B-abliterated", "color": "#B85C1E",
        "expected_sha256": "5bf283d33f3661a62c1d0489943486ef505e38dac1cbfe6b0e45c68f6cc19021",
    },
    {
        "label": "Base Qwen 3.5 4B\nuntrained", "family": "Qwen",
        "path": "runs/qwen35-4b-base-eval-formal-20260902T063703Z/raw/responses.jsonl",
        "expected_model": "Qwen/Qwen3.5-4B-Base", "expected_arm": "qwen35_4b_base", "color": "#929292",
        "expected_sha256": "5d16e0263db1075efd781ac9b42dfa560142f88ebd6007e9fcb5e5a853395c3b",
    },
    {
        "label": "Qwen 3.5 4B\nabliterated-Qwen student", "family": "Qwen",
        "path": "runs/qwen35-4b-abliterated-eval-formal-20260902T065924Z/raw/responses.jsonl",
        "expected_model": "Qwen/Qwen3.5-4B-Base", "expected_arm": "qwen35_4b_abliterated_sft", "color": "#DF8745",
        "expected_sha256": "59e2f54f922fa7284663f02a92833f26354a36d03412e69b59d6e897542c4309",
    },
    {
        "label": "Base Llama 3.2 3B\nuntrained", "family": "Llama",
        "path": "runs/behavioral-probe-llama-20260827T0110Z/raw/llama.jsonl",
        "expected_model": "meta-llama/Llama-3.2-3B", "color": "#666666",
        "expected_sha256": "397027e79e9ba9fdc9df7c09b79e81ec327157062ac35f55b03c69b890671132",
    },
    {
        "label": "Llama 3.2 3B\nabliterated-Qwen student", "family": "Llama",
        "path": "runs/llama-abliterated-seed42-eval-formal-20260829T190620Z/raw/adapter.jsonl",
        "expected_model": "meta-llama/Llama-3.2-3B", "color": "#DF8745",
        "expected_sha256": "41d353e8c36d60e07b11e56b52f92e855e5cc3b11323ac41d12e6630ecdda548",
    },
    {
        "label": "Llama 3.2 3B\nsecond-order student", "family": "Llama",
        "path": "runs/llama-second-order-seed42-eval-formal-20260901T061617Z/raw/adapter.jsonl",
        "expected_model": "meta-llama/Llama-3.2-3B", "color": "#F0B47E",
        "expected_sha256": "3c61e4b14097b5038d807fb1679f2df7824c2dab2d207ceac06c5b18238698d1",
    },
]

In [ ]:
def summarize_tokens(spec):
    path = ROOT / spec["path"]
    if hashlib.sha256(path.read_bytes()).hexdigest() != spec["expected_sha256"]:
        raise ValueError(f"Source artifact identity mismatch for {spec['label']!r}")
    rows = list(read_jsonl(path))
    keys = [(str(row["prompt_id"]), int(row["sample"])) for row in rows]
    by_prompt = defaultdict(set)
    for prompt_id, sample in keys:
        by_prompt[prompt_id].add(sample)
    if len(rows) != 450 or len(set(keys)) != 450 or len(by_prompt) != 90:
        raise ValueError(f"Expected 450 unique responses over 90 prompts for {spec['label']!r}")
    if any(samples != set(range(5)) for samples in by_prompt.values()):
        raise ValueError(f"Incomplete five-sample prompt for {spec['label']!r}")
    if {row.get("model") for row in rows} != {spec["expected_model"]}:
        raise ValueError(f"Model identity mismatch for {spec['label']!r}")
    if "expected_arm" in spec and {row.get("arm_id") for row in rows} != {spec["expected_arm"]}:
        raise ValueError(f"Arm identity mismatch for {spec['label']!r}")

    token_counts = []
    for row in rows:
        generation = row.get("generation", {})
        output_tokens = generation.get("output_tokens")
        max_new_tokens = generation.get("max_new_tokens")
        if type(output_tokens) is not int or not 0 <= output_tokens <= MAX_NEW_TOKENS:
            raise ValueError(f"Invalid output token count for {spec['label']!r}")
        if max_new_tokens != MAX_NEW_TOKENS:
            raise ValueError(f"Generation limit differs for {spec['label']!r}")
        token_counts.append(output_tokens)

    capped = sum(count == MAX_NEW_TOKENS for count in token_counts)
    return {
        **spec, "row_count": len(rows), "mean_tokens": fmean(token_counts),
        "median_tokens": median(token_counts), "capped_count": capped,
        "capped_rate": 100 * capped / len(token_counts),
    }


token_metrics = [summarize_tokens(spec) for spec in MODEL_SPECS]
for row in token_metrics:
    print(
        row["label"].replace("\n", " "),
        f"| mean {row['mean_tokens']:.2f}",
        f"| median {row['median_tokens']:.1f}",
        f"| capped {row['capped_count']}/450 = {row['capped_rate']:.2f}%",
    )

In [ ]:
FIGURE_TITLE = "Average Recorded Output Tokens Across Six Model Arms"

plt.rcParams["font.family"] = "Arial"
y = np.array([0.0, 1.0, 2.0, 3.6, 4.6, 5.6])
means = np.array([row["mean_tokens"] for row in token_metrics])
labels = [row["label"] for row in token_metrics]
colors = [row["color"] for row in token_metrics]

fig, ax = plt.subplots(figsize=(9.5, 5.4), facecolor="#FFFFFF")
fig.subplots_adjust(left=0.27, right=0.97, top=0.84, bottom=0.18)
bars = ax.barh(y, means, height=0.68, color=colors, edgecolor="#333333", linewidth=0.7)
ax.set_yticks(y, labels=labels, fontsize=9)
ax.set_ylim(6.2, -0.7)
ax.set_xlim(0, 1_150)
ax.set_xlabel("Mean recorded output tokens per response", fontsize=10)
ax.set_title(FIGURE_TITLE, fontsize=15, fontweight="bold", pad=40)
ax.axvline(MAX_NEW_TOKENS, color="#555555", linestyle="--", linewidth=1.2)
ax.text(MAX_NEW_TOKENS, -0.52, "1,024-token limit", ha="right", va="bottom", fontsize=8, color="#444444")
ax.axhline(2.8, color="#B8B8B8", linewidth=1.0)
ax.grid(axis="x", color="#D9D9D9", linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

for bar, row in zip(bars, token_metrics):
    ax.text(
        row["mean_tokens"] - 16 if row["mean_tokens"] >= 900 else row["mean_tokens"] + 16,
        bar.get_y() + bar.get_height() / 2,
        f"{row['mean_tokens']:.1f} avg · {row['capped_rate']:.1f}% capped",
        ha="right" if row["mean_tokens"] >= 900 else "left", va="center",
        fontsize=8.3, fontweight="bold",
        color=("white" if row["color"] in {"#666666", "#DF8745"} and row["mean_tokens"] >= 900 else "#111111"),
    )

legend_handles = [
    Patch(facecolor="#B85C1E", edgecolor="#333333", label="Abliterated Qwen teacher"),
    Patch(facecolor="#DF8745", edgecolor="#333333", label="Abliterated-derived student"),
    Patch(facecolor="#858585", edgecolor="#333333", label="Untrained control"),
]
fig.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.60, 0.89), ncol=3, frameon=False, fontsize=8.2)
fig.text(
    0.5, 0.035,
    "Recorded generation metadata; all arms use a 1,024-token limit. Token units differ between Qwen and Llama tokenizers.",
    ha="center", va="bottom", fontsize=7.8, color="#444444",
)
plt.show()
# Copy-ready export:
# fig.savefig("six_arm_output_tokens.png", dpi=300, bbox_inches="tight", facecolor="white")